In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import soundfile as sf
from dataclasses import dataclass
from typing import Any, Dict, List
from datasets import Dataset, DatasetDict
from transformers import (
    SpeechT5ForTextToSpeech, 
    SpeechT5Processor, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

# 1. CONFIGURATION
DATA_DIR = r"C:\TTS-Speech_t5\XTTS_Dataset"
TRAIN_METADATA = r"C:\TTS-Speech_t5\XTTS_Dataset\train.csv"
TEST_METADATA = r"C:\TTS-Speech_t5\XTTS_Dataset\validation.csv"
CHECKPOINT_PATH = r"C:\TTS-Speech_t5\speecht5_tts"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. LOAD MODELS
print(f"Loading model and processor to {device}...")
processor = SpeechT5Processor.from_pretrained(CHECKPOINT_PATH)
model = SpeechT5ForTextToSpeech.from_pretrained(CHECKPOINT_PATH).to(device)

# 3. DATA LOADING & FILE VALIDATION
def load_and_validate(path):
    df = pd.read_csv(path, header=None, sep='|', names=['file_name', 'text'])
    df['file_name'] = df['file_name'].astype(str).str.strip()
    
    def get_full_path(fname):
        if not fname.lower().endswith(".wav"):
            fname += ".wav"
        return os.path.join(DATA_DIR, fname)
    
    df['full_path'] = df['file_name'].apply(get_full_path)
    initial_count = len(df)
    df = df[df['full_path'].apply(os.path.exists)]
    
    print(f"Loaded {len(df)}/{initial_count} valid files from {os.path.basename(path)}")
    return df

df_train = load_and_validate(TRAIN_METADATA)
df_test = load_and_validate(TEST_METADATA)

dataset = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "test": Dataset.from_pandas(df_test)
})

# --- 4. PREPROCESSING ---
def prepare_dataset(example):
    # Manual load using soundfile (Bypasses torchcodec)
    speech, sr = sf.read(example["full_path"])
    
    # Ensure Mono
    if len(speech.shape) > 1:
        speech = np.mean(speech, axis=1)

    # SpeechT5 Feature Extraction
    inputs = processor(
        text=example["text"], 
        audio_target=speech,
        sampling_rate=16000,
        return_attention_mask=False,
    )
    
    # Return as list for Dataset compatibility
    return {
        "input_ids": inputs["input_ids"],
        "labels": inputs["labels"],
        "speaker_embeddings": np.zeros(512, dtype=np.float32).tolist() 
    }

print("Preprocessing dataset (Converting audio to Mel-spectrograms)...")
final_dataset = dataset.map(
    prepare_dataset, 
    remove_columns=dataset["train"].column_names,
    desc="Processing"
)

# 5. DATA COLLATOR (Manual Padding Logic)
@dataclass
class DataLoaderSpeechT5WithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # 1. Pad Text (Input IDs)
        input_ids = [{"input_ids": feature["input_ids"]} for feature in features]
        batch = self.processor.tokenizer.pad(input_ids, return_tensors="pt")

        # 2. Pad Audio (Labels)
        # Force shape to (Sequence_Length, 80) to avoid dimension mismatch
        label_features = [torch.tensor(feature["labels"]).view(-1, 80) for feature in features]
        
        labels_padded = torch.nn.utils.rnn.pad_sequence(
            label_features, 
            batch_first=True, 
            padding_value=0.0
        )
        
        # Reduction Factor Fix: SpeechT5 needs even length for the decoder
        if labels_padded.shape[1] % 2 != 0:
            labels_padded = torch.nn.functional.pad(labels_padded, (0, 0, 0, 1))
            
        batch["labels"] = labels_padded

        # 3. Handle Speaker Embeddings
        speaker_embeddings = np.array([feature["speaker_embeddings"] for feature in features])
        batch["speaker_embeddings"] = torch.tensor(speaker_embeddings, dtype=torch.float32)

        return batch

data_collator = DataLoaderSpeechT5WithPadding(processor=processor)

# 6. TRAINING ARGUMENTS
training_args = Seq2SeqTrainingArguments(
    output_dir="./speecht5_finetuned_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=5000,
    save_steps=100,
    eval_strategy="steps",
    save_total_limit=2,
    fp16=True if device == "cuda" else False,
    label_names=["labels"],
    report_to="none",
    continue_path="C:\TTS-Speech_t5\speecht5_finetuned_results\checkpoint-1000"
)

#7. TRAIN
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["test"],
    data_collator=data_collator,
)



print("Setup complete. Starting training loop...")
trainer.train()

c:\Users\amawg\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model and processor to cuda...


Loading weights: 100%|██████████| 396/396 [00:00<00:00, 663.77it/s, Materializing param=speecht5.encoder.wrapped_encoder.layers.11.layer_norm.weight]                     
SpeechT5ForTextToSpeech LOAD REPORT from: C:\TTS-Speech_t5\speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 5756/5757 valid files from train.csv
Loaded 1016/1017 valid files from validation.csv
Preprocessing dataset (Converting audio to Mel-spectrograms)...


Processing: 100%|██████████| 1016/1016 [00:26<00:00, 37.66 examples/s]


TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'continue_path'

In [1]:
import torch
import transformers
import speechbrain
import soundfile

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Transformers version: {transformers.__version__}")
print(f"SpeechBrain version: {speechbrain.__version__}")

c:\Users\amawg\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA Available: True
Transformers version: 5.2.0
SpeechBrain version: 1.0.3
